In [1]:
import os
import re
import pandas as pd
from langchain_ollama import OllamaLLM
from langchain.prompts import ChatPromptTemplate
import pandas as pd

metadata_filtered_df = pd.read_csv("../../data/preprocessed/amazon-Video_Games/metadata_filtered_df.csv")

In [40]:
# Fix the import error by using the correct class name
from transformers import pipeline, AutoTokenizer, AutoModelForSeq2SeqLM
import torch
import time

# Load a text generation model (T5 or similar for text-to-text tasks)
model_name = "t5-small"  # You can also use "facebook/bart-base" or "google/flan-t5-small"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

# Create a text generation pipeline
text_generator = pipeline(
    "text2text-generation",
    model=model,
    tokenizer=tokenizer,
    max_length=1000,
    do_sample=True,
    temperature=0.7
)

# Function to generate description for a single product
def generate_description(product_text):
    try:
        # Create a prompt for the model
        prompt = f"summarize: {product_text[:1000]}"  # Limit input length
        
        # Generate the description
        response = text_generator(prompt, max_length=1000, num_return_sequences=1)
        
        return response[0]['generated_text'].strip()
    except Exception as e:
        print(f"Error generating description: {e}")
        return "Description generation failed."


print("Generating descriptions for sample products...")
descriptions = []

for idx, row in metadata_filtered_df.iterrows():
    print(f"Processing product {idx + 1}/{len(metadata_filtered_df)}")
    description = generate_description(row['source_text'])
    descriptions.append(description)

# Add the generated descriptions to the dataframe
metadata_filtered_df['generated_description'] = descriptions

Hardware accelerator e.g. GPU is available in the environment, but no `device` argument is passed to the `Pipeline` object. Model will be on CPU.


Generating descriptions for sample products...
Processing product 1/19829
Processing product 2/19829
Processing product 3/19829
Processing product 4/19829
Processing product 5/19829
Processing product 6/19829
Processing product 7/19829
Processing product 8/19829
Processing product 9/19829
Processing product 10/19829
Processing product 11/19829
Processing product 12/19829
Processing product 13/19829
Processing product 14/19829
Processing product 15/19829
Processing product 16/19829
Processing product 17/19829
Processing product 18/19829
Processing product 19/19829
Processing product 20/19829
Processing product 21/19829
Processing product 22/19829
Processing product 23/19829
Processing product 24/19829
Processing product 25/19829
Processing product 26/19829
Processing product 27/19829
Processing product 28/19829
Processing product 29/19829
Processing product 30/19829
Processing product 31/19829
Processing product 32/19829
Processing product 33/19829
Processing product 34/19829
Processing

In [44]:
metadata_filtered_df.to_csv("/Users/U725801/Documents/GitHub/Masterarbeit-Playground/data/preprocessed/amazon-Video_Games/metadata_filtered_df.csv")

Hybrid Hearst with pattern-based & embedding based

In [5]:
import time 
import tqdm
import json 
import numpy as np 
import pandas as pd 
import spacy
from spacy.matcher import PhraseMatcher,Matcher
from spacy.tokens import Span
from spacy.tokens import Token, DocBin

from collections import Counter
from nltk.tokenize import MWETokenizer
from nltk.util import Trie
import nltk




In [28]:
# Here lowercase=False option is used to keep the original case of the terms, since we possibly could have term abbreviations. Like API, CAT, etc.
from sklearn.feature_extraction.text import CountVectorizer
vocabulary = metadata_filtered_df['source_text'].iloc[0:int((len(metadata_filtered_df)*0.2))].str.split().explode().unique().tolist()
cvectorizer = CountVectorizer(ngram_range=(
    1, 4), stop_words="english", vocabulary=vocabulary, lowercase=True)
X = cvectorizer.fit_transform(metadata_filtered_df['source_text'].iloc[0:int((len(metadata_filtered_df)*0.2))])

# Show top-25 most frequent terms
termdf_cv = pd.DataFrame(np.sum(X, axis=0), columns=cvectorizer.get_feature_names_out(
)).T.sort_values(by=0, ascending=False)
termdf_cv.head(50)

/Users/U725801/Library/Python/3.12/lib/python/site-packages/sklearn/feature_extraction/text.py:1364: UserWarning: Upper case characters found in vocabulary while 'lowercase' is True. These entries will not be matched with any documents
  warnings.warn(


,0
item,9904
game,9322
price,8783
games,8137
store,8067
manufacturer,6939
video,6490
date,6279
list,5987
ounces,5408


Looking at the top 50 used words in the source_text column we notice that most of items are not of the type entity rather ounces are mostly stop words. Additionally we try to find a portable framework without necceseity for pre-training. Therefore we process the text with a base uncased model, if the accuracy is poorly we will come back to pretraining the model with transfer-learning on a different dataset

In [6]:
rypher_patterns ={ 
    "rhyper-multi" : [
        [ 
            {"LOWER": {"IN": ["features", "properties"]}, "OP":"!"},
            {"ENT_TYPE": "ENTITY"},
            {"ORTH": ",", "OP": "?"},
            {"LOWER": "such"},
            {"LOWER": "as"},
            {"LOWER": {"IN": ["a", "an","the"]}, "OP":"?"},
            {"ENT_TYPE": "ENTITY"}
        ],
        [ 
            {"ENT_TYPE": "ENTITY"},
            {"ORTH": ",", "OP": "?"},
            {"LOWER": "including"},
            {"LOWER": {"IN": ["a", "an","the"]}, "OP":"?"},
            {"ENT_TYPE": "ENTITY"} 
        ]
    ],

    "hyper-single" : [
        [
            {"LOWER": {"IN": ["unlike", "like"]}},
            {"LOWER": {"IN": ["most", "all", "any", "other"]}},
            {"LOWER": {"IN": ["a", "an","the"]}, "OP":"?"},
            {"ENT_TYPE": "ENTITY"},
            {"ORTH": ","},
            {"LOWER": {"IN": ["a", "an","the"]}, "OP":"?"},
            {"ENT_TYPE": "ENTITY"}
        ]
    ],

    "rhyper-single" : [
        [ 
            {"ENT_TYPE": "TECH"},
            {"LOWER": "which"},
            {"LOWER": {"IN": ["is", "are"]}},
            {"LOWER": {"IN": ["a", "an"]}},
            {"LOWER": {"IN": ["example", "class","kind"]}},
            {"LOWER": "of"},
            {"LOWER": {"IN": ["a", "an","the"]}, "OP":"?"},
            {"ENT_TYPE": "TECH"}
        ],
        [ 
            {"ENT_TYPE": "TECH"},
            {"LOWER": {"IN": ["and", "or"]}},
            {"LOWER": {"IN": ["any", "some"]}},
            {"LOWER": "other"},
            {"LOWER": {"IN": ["a", "an","the"]}, "OP":"?"},
            {"ENT_TYPE": "TECH"}
        ],
        [
            {"ENT_TYPE": "ENTITY"},
            {"ORTH": ",", "OP": "?"},
            {"LOWER": "which"},
            {"LOWER": {"IN": ["is", "are"]}},
            {"LOWER": {"IN": ["also", "sometimes"]}, "OP":"?"},
            {"LOWER": "called"},
            {"LOWER": {"IN": ["a", "an","the"]}, "OP":"?"},
            {"ENT_TYPE": "ENTITY"}
        ],
        [ 
            {"ENT_TYPE": "ENTITY"},
            {"LOWER": "a"},
            {"LOWER": "special"},
            {"LOWER": "case"},
            {"LOWER": "of"},
            {"LOWER": {"IN": ["a", "an","the"]}, "OP":"?"},
            {"ENT_TYPE": "TECH"}
        ],
        [ 
            {"ENT_TYPE": "ENTITY"},
            {"LOWER": {"IN": ["is", "are"]}},
            {"LOWER": {"IN": ["a", "an","the"]}},
            {"ENT_TYPE": "ENTITY"},
            {"LOWER": "that"}
        ],
        [
            {"ENT_TYPE": "ENTITY"},
            {"LOWER": {"IN": ["is", "are"]}},
            {"LOWER": {"IN": ["a", "an","the"]}},
            {"ENT_TYPE": "ENTITY"}
        ]   
    ]
}

In [21]:
import sys
!{sys.executable} -m pip install spacy
!{sys.executable} -m pip install spacy-transformers

!{sys.executable} -m spacy download en_core_web_lg

Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 25.0.1 -> 25.1.1
[notice] To update, run: pip3 install --upgrade pip
Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 25.0.1 -> 25.1.1
[notice] To update, run: pip3 install --upgrade pip
Defaulting to user installation because normal site-packages is not writeable
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 587.7/587.7 MB 50.4 MB/s eta 0:00:0000:0100:01

[notice] A new release of pip is available: 25.0.1 -> 25.1.1
[notice] To update, run: pip3 install --upgrade pip
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_lg')


In [ ]:
metadata_filtered_df['source_text'].iloc[20]

In [30]:
import spacy
nlp_model = spacy.load("en_core_web_lg")


doc = nlp_model(metadata_filtered_df['source_text'].iloc[20])

colors = {"ORG": "#F67DE3"}
options = {"colors": colors}

spacy.displacy.render(doc, style="ent", options=options, jupyter=True)

In [61]:
class Hearst_Patterns:
    """ Extracts hearst patterns from a pandas DataFrame containing product metadata
    """

    def __init__(self, patterns_dict=None, model_name="en_core_web_lg"):
        """ creates an instance of the class Hearst_Patterns

        Args:
            patterns_dict (dict, optional): dictionary containing the patterns. Defaults to None.
            model_name (str, optional): the spacy model to use. Defaults to "en_core_web_lg".
        """
        # load the spacy model
        self.nlp = spacy.load(model_name)
        
        # add pipes if they don't exist
        if "merge_entities" not in self.nlp.pipe_names:
            self.nlp.add_pipe("merge_entities")
        if "merge_noun_chunks" not in self.nlp.pipe_names:
            self.nlp.add_pipe('merge_noun_chunks')

        # initialize matcher
        self.matcher = Matcher(self.nlp.vocab)
        
        # load patterns
        if patterns_dict:
            self.patterns = patterns_dict
            self._load_patterns_to_matcher()
        
        # this list is used in the method get_matches
        self.continue_words = [',', 'and', 'or', ';', 'also', 'as well']

    def _load_patterns_to_matcher(self):
        """ Load patterns from dictionary into the matcher """
        for pattern_name, pattern_list in self.patterns.items():
            for i, pattern in enumerate(pattern_list):
                self.matcher.add(f"{pattern_name}-{i}", pattern)

    def extract_entity_relations_from_dataframe(self, df, text_column="generated_description", asin_column="parent_asin", size=None):
        """ Extract entity relations from a pandas DataFrame using NER
        
        Args:
            df (pd.DataFrame): DataFrame containing the text data and ASIN
            text_column (str, optional): name of the column containing text. Defaults to "generated_description".
            asin_column (str, optional): name of the column containing ASIN. Defaults to "asin".
            size (int, optional): maximum number of texts to process. Defaults to None (process all).

        Returns:
            list: list of extracted entity relations as tuples (head, relation, tail, text)
        """
        extracted_relations = []
        
        # limit the number of texts to process if size is specified
        texts_to_process = df[[text_column, asin_column]].iloc[:size] if size else df[[text_column, asin_column]]
        
        print(f"Processing {len(texts_to_process)} texts for entity relations...")
        
        for idx, row in texts_to_process.iterrows():
            text = row[text_column]
            asin = row[asin_column]
            
            if pd.isna(text) or text.strip() == "":
                continue
                
            try:
                relations = self.get_entity_relations(text, asin)
                if relations:
                    extracted_relations.extend(relations)
                    
                if idx % 100 == 0:
                    print(f"Processed {idx}/{len(texts_to_process)} texts, found {len(extracted_relations)} entity relations")
                    
            except Exception as e:
                print(f"Error processing text {idx}: {e}")
                continue

        print(f"Total entity relations extracted: {len(extracted_relations)}")


        # list to df 
        ner_relations_df = pd.DataFrame(extracted_relations, columns=['head', 'relation', 'tail', 'label'])
        # Rename columns to match the expected structure: head=parent_asin, relation='related to', tail=ner_entity
        ner_relations_df = ner_relations_df.rename(columns={'word1': 'head', 'word2': 'tail'})
        ner_relations_df['relation'] = 'related to'  # Set all relations to 'related to'
        ner_relations_df = ner_relations_df.drop_duplicates()

        return ner_relations_df

    def get_entity_relations(self, text, asin):
        """ Extract entity relations from a single text using NER

        Args:
            text (str): text to analyze
            asin (str): ASIN of the product

        Returns:
            list: list of extracted entity relations as tuples (head, relation, tail, text)
        """
        doc = self.nlp(text)
        relations = []
        
        # Extract all named entities from the text
        for ent in doc.ents:
            entity_text = ent.text.strip()
            entity_label = ent.label_
            
            # Create hypernym relation: ASIN -> entity (ASIN is parent of entity)
            hypernym_relation = (asin, "related_to", entity_text, text)
            relations.append(hypernym_relation)
            
        
        # Remove duplicates while preserving order
        seen = set()
        unique_relations = []
        for relation in relations:
            if relation not in seen:
                seen.add(relation)
                unique_relations.append(relation)
        
        return unique_relations

    def extract_patterns_from_dataframe(self, df, text_column="generated_description", size=None):
        """ Extract patterns from a pandas DataFrame

        Args:
            df (pd.DataFrame): DataFrame containing the text data
            text_column (str, optional): name of the column containing text. Defaults to "source_text".
            size (int, optional): maximum number of texts to process. Defaults to None (process all).

        Returns:
            list: list of extracted patterns as tuples (word1, word2, relation, label, text)
        """
        extracted_patterns = []
        
        # limit the number of texts to process if size is specified
        texts_to_process = df[text_column].iloc[:size] if size else df[text_column]
        
        print(f"Processing {len(texts_to_process)} texts...")
        
        for idx, text in enumerate(texts_to_process):
            if pd.isna(text) or text.strip() == "":
                continue
                
            try:
                patterns = self.get_matches(text)
                if patterns:
                    extracted_patterns.extend(patterns)
                    
                if idx % 100 == 0:
                    print(f"Processed {idx}/{len(texts_to_process)} texts, found {len(extracted_patterns)} patterns")
                    
            except Exception as e:
                print(f"Error processing text {idx}: {e}")
                continue

        print(f"Total patterns extracted: {len(extracted_patterns)}")
        return extracted_patterns

    def get_matches(self, text):
        """ Extract matches from a single text

        Args:
            text (str): text to analyze

        Returns:
            list: list of extracted relations as tuples (word1, word2, relation, label, text)
        """
        label = {
            'rhyper': -1,
            'hyper': 1,
        }
        
        # add a period at the beginning to handle patterns that don't work at sentence start
        doc = self.nlp('. ' + text)

        matches = self.matcher(doc)
        relations = []
        
        for match_id, start, end in matches:
            # get all entities indices in the doc
            ent_indices = [i for i in range(start, end) if doc[i].text in [
                ent.text for ent in doc[start:end].ents]]
            
            if not ent_indices:  # no entity found
                continue

            # extract X...Y from a match ..X...Y.., so now we know that the first and the last token are the entities
            span = doc[min(ent_indices):max(ent_indices)+1]

            # Get string representation
            match_info = self.nlp.vocab.strings[match_id]
            match_name = match_info.split('-')[0]   # hyper or rhyper
            match_type = match_info.split('-')[1]   # single or multi

            np_0 = span[0]  # left term
            np_1 = span[-1]  # right term (or first right term if multiple)

            # all the right terms (ex. for Y...X1, X2, ...Xn) X1...Xn are the right terms
            right_terms = [np_1.text]
            if match_type == "multi":  # look for other terms (X2,X3..etc)
                # we use the same model to get the noun chunks
                doc_remaining = self.nlp(doc[end:].text)
                for d in doc_remaining:
                    # look for entities inside the noun chunk
                    matching_ents = [
                        ent.text for ent in doc.ents if ent.text in d.text]
                    if matching_ents:
                        right_terms.append(matching_ents[0])
                    elif d.text not in self.continue_words:  # stop when seeing a word that's not in the list
                        break

            for term in right_terms:
                relations.append(
                    (np_0.text, term, match_name, label[match_name], text))

        relations = set(relations)
        return list(relations)

    def save_patterns_to_csv(self, patterns, save_path="/Users/U725801/Documents/GitHub/Masterarbeit-Playground/data/taxonomy/amazon-Video_Games/hybrid_hearst_patterns.csv"):
        """ Save extracted patterns to CSV file

        Args:
            patterns (list): list of extracted patterns
            save_path (str, optional): path to save the CSV file. Defaults to "hearst_patterns.csv".
        """
        df = pd.DataFrame(patterns, columns=['word1', 'word2', 'relation', 'label', 'text'])
        df.to_csv(save_path, index=False)
        print(f"Patterns saved to {save_path}")
        return df

# call the class
hearst_patterns = Hearst_Patterns()
patterns = hearst_patterns.extract_patterns_from_dataframe(metadata_filtered_df)
hearst_patterns.save_patterns_to_csv(patterns)

# with just ner
ner_relations = hearst_patterns.extract_entity_relations_from_dataframe(metadata_filtered_df)
ner_relations.to_csv("/Users/U725801/Documents/GitHub/Masterarbeit-Playground/data/taxonomy/amazon-Video_Games/ner_relations.csv", index=False)

Processing 19829 texts...


/var/folders/33/fw5wlwpj4b78_xvtgsm72q6r0000gq/T/ipykernel_43807/1861851452.py:175: UserWarning: [W036] The component 'matcher' does not have any patterns defined.
  matches = self.matcher(doc)


Processed 0/19829 texts, found 0 patterns
Processed 100/19829 texts, found 0 patterns
Processed 200/19829 texts, found 0 patterns
Processed 300/19829 texts, found 0 patterns
Processed 400/19829 texts, found 0 patterns
Processed 500/19829 texts, found 0 patterns
Processed 600/19829 texts, found 0 patterns
Processed 700/19829 texts, found 0 patterns
Processed 800/19829 texts, found 0 patterns
Processed 900/19829 texts, found 0 patterns
Processed 1000/19829 texts, found 0 patterns
Processed 1100/19829 texts, found 0 patterns
Processed 1200/19829 texts, found 0 patterns
Processed 1300/19829 texts, found 0 patterns
Processed 1400/19829 texts, found 0 patterns
Processed 1500/19829 texts, found 0 patterns
Processed 1600/19829 texts, found 0 patterns
Processed 1700/19829 texts, found 0 patterns
Processed 1800/19829 texts, found 0 patterns
Processed 1900/19829 texts, found 0 patterns
Processed 2000/19829 texts, found 0 patterns
Processed 2100/19829 texts, found 0 patterns
Processed 2200/19829 t

We use patterns with patterns with single and co-hyponym patterns. This strategy showed good results in boosting recall in hybrid approaches for hearst-patterns recognition. [[Ref. Anthology]](https://aclanthology.org/2023.starsem-1.18.pdf#:~:text=BERT%20and%20the%20limitations%20of,sets%20of%20rare%20and%20abstract)

NER RE

In [2]:
import re 

def create_source_text(product):
        """Concatenate product information into a text string

        Args:
            product (dict): dictionary containing product information

        Returns:
            str: concatenated product information
        """
        description = "*" if product["description"] == "" else f"Description: {product['description']}"
        features = "*" if product["features"] == "" else f"Features: {product['features']}"
        details = "*" if product["details"] == "" else f"Details: {product['details']}"
        store = "*" if product["store"] == "" else f"Store: {product['store']}"
        categories = "*" if product["categories"] == "" else f"Categories: {product['categories']}"
        price = "*" if product["price"] == "" else f"Price: {product['price']}"
        author = "*" if product["author"] == "" else f"Author: {product['author']}"

        # concatenated_text = f"Parent ASIN: {product['parent_asin']}; Title: {product['title']}; Author: {author}; Description: {description}; Features: {features} - {details}; Store: {store}; Categories: {categories}; Price: {price};"
        concatenated_text = f"Parent ASIN: {product['parent_asin']}; Title: {product['title']}Description: {description};- {details}; Store: {store};"
        #print("Concatenated text:", concatenated_text)  # Debug: Print the output text
        return concatenated_text
    
def clean_string(string):
            string = re.sub(r'\[', '', string)
            string = re.sub(r'\]', '', string)
            string = re.sub(r'"', '', string)
            string = re.sub(r'\s+', ' ', string)
            string = re.sub("{", "", string)
            string = re.sub("}", "", string)
            return string

def clean_column(df, column_name):
    """
    Clean a column in the dataframe by applying clean_string to each element.
    Handles lists by converting them to strings first.
    
    Args:
        df (pandas.DataFrame): The dataframe containing the column to clean
        column_name (str): The name of the column to clean
        
    Returns:
        pandas.Series: The cleaned column
    """
    def safe_clean(value):
        if isinstance(value, list):
            # Convert list to string before cleaning
            return clean_string(str(value))
        elif pd.isna(value) or value is None:
            return ""
        else:
            return clean_string(str(value))
    
    return df[column_name].apply(safe_clean)

# Clean text columns that might contain lists
for column in ['details', 'features', 'categories', 'description', 'bought_together']:
    if column in metadata_filtered_df.columns:
        metadata_filtered_df[column] = clean_column(metadata_filtered_df, column)

metadata_filtered_df['source_text'] = metadata_filtered_df.apply(create_source_text, axis=1)


In [43]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
from tqdm import tqdm
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
def extract_triplets(text):
    triplets = []
    relation, subject, relation, object_ = '', '', '', ''
    text = text.strip()
    current = 'x'
    for token in text.replace("<s>", "").replace("<pad>", "").replace("</s>", "").split():
        if token == "<triplet>":
            current = 't'
            if relation != '':
                triplets.append({'head': subject.strip(), 'type': relation.strip(),'tail': object_.strip()})
                relation = ''
            subject = ''
        elif token == "<subj>":
            current = 's'
            if relation != '':
                triplets.append({'head': subject.strip(), 'type': relation.strip(),'tail': object_.strip()})
            object_ = ''
        elif token == "<obj>":
            current = 'o'
            relation = ''
        else:
            if current == 't':
                subject += ' ' + token
            elif current == 's':
                object_ += ' ' + token
            elif current == 'o':
                relation += ' ' + token
    if subject != '' and relation != '' and object_ != '':
        triplets.append({'head': subject.strip(), 'type': relation.strip(),'tail': object_.strip()})
    return triplets

# Load model and tokenizer
tokenizer = AutoTokenizer.from_pretrained("Babelscape/rebel-large")
model = AutoModelForSeq2SeqLM.from_pretrained("Babelscape/rebel-large")
gen_kwargs = {
    "max_length": 256,
    "length_penalty": 0,
    "num_beams": 3,
    "num_return_sequences": 3,
}

triples = []

def generate_triples(model,tokenizer,row):
    texts = [row.source_text]
    parent_asin = row.parent_asin
    # Tokenizer text
    model_inputs = tokenizer(texts, max_length=512, padding=True, truncation=True, return_tensors='pt')
    generated_tokens = model.generate(
        model_inputs["input_ids"].to(model.device),
        attention_mask=model_inputs["attention_mask"].to(model.device),
        **gen_kwargs
    )
    decoded_preds = tokenizer.batch_decode(generated_tokens, skip_special_tokens=False)
    for idx, sentence in enumerate(decoded_preds):
        et = extract_triplets(sentence)
        for t in et:
            # link product to head / hyponym
            triples.append((parent_asin, t['type'], t['head']))
            # link head / hyponym to tail / hypernym
            triples.append((t['head'], t['type'], t['tail']))

for i in tqdm(range(0, len(metadata_filtered_df))):
  
    generate_triples(model,tokenizer,metadata_filtered_df.iloc[i])

distinct_triples = list(set(triples))





100%|██████████| 324/324 [12:40<00:00,  2.35s/it]


In [41]:
class NERPatternExtractor:
    def __init__(self,model_name,tokenizer_name):
        self.tokenizer = AutoTokenizer.from_pretrained(tokenizer_name)
        self.model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

    def extract_triplets(text):
        triplets = []
        #TODO: change triplets list to dataframe
        relation, subject, relation, object_ = '', '', '', ''
        text = text.strip()
        current = 'x'
        for token in text.replace("<s>", "").replace("<pad>", "").replace("</s>", "").split():
            if token == "<triplet>":
                current = 't'
                if relation != '':
                    triplets.append({'head': subject.strip(), 'type': relation.strip(),'tail': object_.strip()})
                    relation = ''
                subject = ''
            elif token == "<subj>":
                current = 's'
                if relation != '':
                    triplets.append({'head': subject.strip(), 'type': relation.strip(),'tail': object_.strip()})
                object_ = ''
            elif token == "<obj>":
                current = 'o'
                relation = ''
            else:
                if current == 't':
                    subject += ' ' + token
                elif current == 's':
                    object_ += ' ' + token
                elif current == 'o':
                    relation += ' ' + token
        if subject != '' and relation != '' and object_ != '':
            triplets.append({'head': subject.strip(), 'type': relation.strip(),'tail': object_.strip()})
        return triplets

    def generate_triples(self,row, triples_df):
        texts = [row.source_text]
        parent_asin = row.parent_asin
        # Tokenizer text
        model_inputs = self.tokenizer(texts, max_length=512, padding=True, truncation=True, return_tensors='pt')
        generated_tokens = self.model.generate(
            model_inputs["input_ids"].to(self.model.device),
            attention_mask=model_inputs["attention_mask"].to(self.model.device),
            **gen_kwargs
        )
        decoded_preds = tokenizer.batch_decode(generated_tokens, skip_special_tokens=False)
        for idx, sentence in enumerate(decoded_preds):
            et = extract_triplets(sentence)
            for t in et:
                # link product to head / hyponym
                triples_df = pd.concat([triples_df,pd.DataFrame([{"head":parent_asin,"relation": t['type'], "tail": t['head']}])], ignore_index=True)
                # link head / hyponym to tail / hypernym
                triples_df = pd.concat([triples_df,pd.DataFrame([{"head":t['head'],"relation": t['type'], "tail": t['tail']}])], ignore_index=True)
        return triples_df
if __name__ == "__main__":
    DATASETS = ["Books", "All_Beauty", "Beauty_and_Personal_Care"]
    DATASET = DATASETS[1]
    DIR_NAME = "amazon-" # else: Last_fm, MovieLens

    NER_MODEL = "Babelscape/rebel-large"
    
    # get data
    current_dir = os.getcwd()
    data_path = os.path.join(current_dir, 'data', 'preprocessed', f'{DIR_NAME}{DATASET}', 'metadata_filtered_df.csv')
    metadata_filtered_df = pd.read_csv(data_path)


    # build triples dataframe
    triples_df = pd.DataFrame(columns=["head","relation", "tail"])

    ner_pattern_extractor = NERPatternExtractor(model_name=NER_MODEL,tokenizer_name=NER_MODEL)
    for i in tqdm(range(0, len(metadata_filtered_df))):
        triples_df = ner_pattern_extractor.generate_triples(metadata_filtered_df.iloc[i],triples_df)

    # drop duplicates
    triples_df = triples_df.drop_duplicates()

    # save triples to csv
    out_path = os.path.join(current_dir, 'data', 'relations', f'{DIR_NAME}{DATASET}/ner_relations.csv')
    os.makedirs(os.path.dirname(out_path), exist_ok=True)
    triples_df.to_csv(out_path, index=False)
    print(f"Saved {len(triples_df)} relations to {out_path}")

[('B081ZN3TD5', 'instance of', 'JPNK'),
 ('Spray Bottles', 'use', 'Spray'),
 ('B089CSF11Y', 'use', 'Spray Bottles'),
 ('JPNK 4PCS Anti-Static Detangling Fine & Wide Tooth Shower Comb Set',
  'manufacturer',
  'JPNK'),
 ('B089CSF11Y', 'instance of', 'Cherry'),
 ('JPNK', 'instance of', 'Brand'),
 ('B08LYT4Q2X', 'subclass of', 'Moisturizing'),
 ('B081ZN3TD5', 'manufacturer', 'JPNK 4PCS'),
 ('B08LYT4Q2X', 'part of', 'Moisturizing'),
 ('Moisturizing', 'subclass of', 'Body Oil'),
 ('JPNK 4PCS', 'manufacturer', 'JPNK'),
 ('Cherry', 'instance of', 'Brand'),
 ('B081ZN3TD5',
  'manufacturer',
  'JPNK 4PCS Anti-Static Detangling Fine & Wide Tooth Shower Comb Set'),
 ('Moisturizing', 'part of', 'Skin'),
 ('Moisturizing', 'subclass of', 'Oil')]

hearst_patterns 

In [32]:
class LLMHeirarchy:
    def __init__(self,model):
        self.model = OllamaLLM(model=model,temperature=0) # TODO: in AWS Sagemaker call model via endpoint, api-key

    def chain(self,prompt_type:str,input:str): 

        if prompt_type == "hearst":
            input_message = """
            Create a Hearst pattern for the following text:
            {Text}
            """

            system_prompt = """You are a helpful assistant that can construct Hearst patterns to extract entities and relationships from text.
            You will then need to return the Hearst patterns in a structured format.
            For entities you can use product groups, categories, features, store, ingredients, price range or other suitable entities.
            include not obvious relationships
            if you have information about the product, which are not in the text, you can include them in the hypernym.

            Rules:
            - You will only return the Hearst patterns, dont add notes.
            - Dont include parent_asin, titles in the patterns
            - Build patterns with A -> B, where A is the hypernym and B is the hyponym and B -> C, where B is the hypernym and C is the hyponym
            - Return the Hearst patterns in a structured format. without any regex.
            - Use the commom Hearst Patterns.
            - The output schema is: hypernym("entity"; "hyponym")
            - only use ; to separate the entities.

            Example:
            #1 input: "Harry Potter, The Lord of the Rings, The Prophet, The Alchemist or other books are bestsellers"
            output: hyponym("Harry Potter"; "book"), hyponym("The Lord of the Rings"; "book"), hyponym("The Prophet"; "book"), hyponym("The Alchemist"; "book")

            #2 input: "Harry Potter, The Lord of the Rings, The Prophet, The Alchemist or other books are bestsellers"
            output: hyponym("Harry Potter"; "book"), hyponym("The Lord of the Rings"; "book"), hyponym("The Prophet"; "book"), hyponym("The Alchemist"; "book")

            #3 input: "The book Harry Potter is a fantasy book with a lot of magic especially for teenagers"
            output: hyponym("book"; "Harry Potter"), hyponym("book"; "fantasy book"), hyponym("audience group"; "teenagers")

            #4 
            input: "Most New York Times bestsellers, especially Rich dad poor dad, The 48 laws of power or Atomic habits are self-help books"
            output: hyponym("New York Times bestsellers"; "Rich dad poor dad"), hyponym("New York Times bestsellers"; "The 48 laws of power"), hyponym("New York Times bestsellers"; "Atomic habits")

            #5
            input: "All horror movies are scary including the movie The Conjuring, Saw and The Exorcist"
            output: hyponym("horror movies"; "The Conjuring"), hyponym("horror movies"; "Saw"), hyponym("horror movies"; "The Exorcist")

            #6 
            input: "Babo Botanicals Sheer Zinc Continuous Spray Sunscreen SPF 30 is a sunscreen with Aloe Vera, Sunflower Oil, Baby Skin, Kids Skin, Sensitive Skin, Vegan Product, Mineral Active Ingredient, Water-Resistant Product"
            output: hyponym("Aloe Vera"; "Ingredient"), hyponym("Sunflower Oil"; "Ingredient"), hyponym("Baby Skin"; "Effect"), hyponym("Kids Skin"; "Effect"), hyponym("Sensitive Skin"; "Effect"), hyponym("Vegan Product"; "Sustainable Product"),hyponym("Sustainable Product"; "Product"), hyponym("Mineral Active"; "Feature"), hyponym("Water-Resistant Product"; "Feature")

            #7 
            input: Hypernym: Glycerin Product, Hyponym: Cathy Doll L-Glutathione Magic Cream SPF 50 Whitening Sunscreen 138ml with Glycerin and whitening effect
            output hyponym("Glycerin Product": "Feature"), hyponym("Cathy Doll L-Glutathione Magic Cream SPF 50 Whitening Sunscreen": "Product"), hyponym("Glycerin": "Ingredient"), hyponym("Whitening effect": "Feature")

            #8
            input: "The product is a sunscreen with Aloe Vera, Sunflower Oil, Baby Skin, Kids Skin, Sensitive Skin, Vegan Product, Mineral Active Ingredient, Water-Resistant Product"
            output: hyponym("Aloe Vera"; "Ingredient"), hyponym("Sunflower Oil"; "Ingredient"), hyponym("Baby Skin"; "Effect"), hyponym("Kids Skin"; "Effect"), hyponym("Sensitive Skin"; "Effect"), hyponym("Vegan Product"; "Product"), hyponym("Mineral Active"; "Feature"), hyponym("Water-Resistant Product"; "Feature")

            """

        elif prompt_type == "hierarchical":
            input_message = """
            Create a hierarchical topic model for the following text:
            {Text}
            """
            system_prompt = """
            Role:You are a helpful assistant that extract entities and relationships from text.
            Goal:Your goal is to find entities, relationships from the text.
            The entities should be able to be hierarchically related to each other.
            Make the entities as general as possible, so that they can be used for a hierarchical topic model and found in various contexts.
            You are allowed to use your knowledge about the product, if it is not in the text.
            
            Rules:
            - You will only return the Entities and relationships, dont add notes.
            - Dont include identifiers like parent_asin, titles in the patterns
            - Build patterns with A -> B, where A is the hypernym and B is the hyponym and B -> C, where B is the hypernym and C is the hyponym
            - Return without any regex.
            - The output schema is: "entity-1"("relation"; "entity-2")
            - only use ; to separate the entities.
            - Write all words in lowercase.
            - Use short words for entities and relationships.

            Example:
            #1input: "The product is a sunscreen with Aloe Vera, Sunflower Oil, Baby Skin, Kids Skin, Sensitive Skin, Vegan Product, Mineral Active Ingredient, Water-Resistant Product"
            output: hyponym("product"; "sustainble ingredients"), hyponym("sustainble ingredients"; "aloe vera"), hyponym("sustainble ingredients"; "sunflower oil"), hyponym("Feautures"; "water resistant"), hyponym("Feautures"; "vegan"), hyponym("Feautures"; "mineral active"), hyponym("target audience"; "sensitive skin"), hyponym("target audience"; "kids skin"), hyponym("target audience"; "vegan")

            #2
            input: "The book Harry Potter is a fantasy book with a lot of magic especially for teenagers"
            output: hyponym("Harry Potter"; "fantasy book"), hyponym("fantasy book"; "book"), hyponym("Harry Potter"; "teenagers"), hyponym("Harry Potter"; "j.k. rowling")
            """
        prompt = ChatPromptTemplate(messages=[
    ("system", system_prompt),
    ("user", input_message)
    ], 
            input_variables=["Text"]
        )

        chain = prompt | self.model
        result = chain.invoke({f"Text": input})
        return result
    
    def create_triples(self,df,hypernym,hyponym,parent_asin): 
        df = pd.concat([df,pd.DataFrame([{"head":parent_asin,"relation": "related to", "tail": hyponym}])], ignore_index=True)
        df = pd.concat([df,pd.DataFrame([{"head":hyponym,"relation": "related to", "tail": hypernym}])], ignore_index=True)
        return df


    def post_process(self,result, df, row):
        """
        Extract the hypernym and hyponym from the result and build triples 

        Args:
            result (str): The answer hypernym and hyponym from the LLM
            df (pd.DataFrame): The dataframe to store the triples
            row (pd.Series): The row from the metadata_filtered_df, indicates the current product

        Returns:
            df (pd.DataFrame): The dataframe with the triples
        """
        parent_asin = row.parent_asin
        for line in result.split("\n"):
            if "hyponym" in line:
                match = re.search(r'hyponym\("([^"]+)"\s*;\s*"([^"]+)"\)', line) 
                if match:
                    hypernym, hyponym = match.groups()
                    df = self.create_triples(df,hypernym,hyponym,parent_asin)
        return df
    

if __name__ == "__main__":
    DATASETS = ["Books", "All_Beauty", "Beauty_and_Personal_Care"]
    DATASET = DATASETS[1]
    DIR_NAME = "amazon-" # else: Last_fm, MovieLens
    LLM_MODEL = "llama3.2"
    
    # Get the project root directory (Masterarbeit-Playground folder)
    current_dir = os.getcwd()
    data_path = os.path.join(current_dir, 'data', 'preprocessed', f'{DIR_NAME}{DATASET}', 'metadata_filtered_df.csv')
    metadata_filtered_df = pd.read_csv(data_path)

    # build triples dataframe
    triples_df = pd.DataFrame(columns=["head","relation", "tail"])

    # initialize llm hierarchy
    llm_hierarchy = LLMHeirarchy(model=LLM_MODEL)

# try with iloc 1:10
    for i,row in metadata_filtered_df.iloc[0:3].iterrows():
        result = llm_hierarchy.chain("hierarchical", input=row.source_text)
        print(result)
        print(f"Processing row {i+1} of {len(metadata_filtered_df)}")
        print("--------------------------------")
        
        triples_df = llm_hierarchy.post_process(result,triples_df,row)

    out_path = os.path.join(current_dir, 'data', 'relations', f'{DIR_NAME}{DATASET}','{LLM_MODEL}_triples.csv')
    triples_df.to_csv(out_path, index=False)
 


FileNotFoundError: [Errno 2] No such file or directory: '/Users/U725801/Documents/GitHub/Masterarbeit-Playground/src/playground/data/preprocessed/amazon-All_Beauty/metadata_filtered_df.csv'

In [16]:



input_message = """
Create a Hearst pattern for the following text:
{Text}
"""

system_prompt = """You are a helpful assistant that can construct Hearst patterns to extract entities and relationships from text.
You will then need to return the Hearst patterns in a structured format.
For entities you can use product groups, categories, features, store, ingredients, price range or other suitable entities.
include not obvious relationships
if you have information about the product, which are not in the text, you can include them in the hypernym.

Rules:
- You will only return the Hearst patterns, dont add notes.
- Dont include parent_asin, titles in the patterns
- Build patterns with A -> B, where A is the hypernym and B is the hyponym and B -> C, where B is the hypernym and C is the hyponym
- Return the Hearst patterns in a structured format. without any regex.
- Use the commom Hearst Patterns.
- The output schema is: hypernym("entity"; "hyponym")
- only use ; to separate the entities.

Example:
#1 input: "Harry Potter, The Lord of the Rings, The Prophet, The Alchemist or other books are bestsellers"
output: hyponym("Harry Potter"; "book"), hyponym("The Lord of the Rings"; "book"), hyponym("The Prophet"; "book"), hyponym("The Alchemist"; "book")

#2 input: "Harry Potter, The Lord of the Rings, The Prophet, The Alchemist or other books are bestsellers"
output: hyponym("Harry Potter"; "book"), hyponym("The Lord of the Rings"; "book"), hyponym("The Prophet"; "book"), hyponym("The Alchemist"; "book")

#3 input: "The book Harry Potter is a fantasy book with a lot of magic especially for teenagers"
output: hyponym("book"; "Harry Potter"), hyponym("book"; "fantasy book"), hyponym("audience group"; "teenagers")

#4 
input: "Most New York Times bestsellers, especially Rich dad poor dad, The 48 laws of power or Atomic habits are self-help books"
output: hyponym("New York Times bestsellers"; "Rich dad poor dad"), hyponym("New York Times bestsellers"; "The 48 laws of power"), hyponym("New York Times bestsellers"; "Atomic habits")

#5
input: "All horror movies are scary including the movie The Conjuring, Saw and The Exorcist"
output: hyponym("horror movies"; "The Conjuring"), hyponym("horror movies"; "Saw"), hyponym("horror movies"; "The Exorcist")

#6 
input: "Babo Botanicals Sheer Zinc Continuous Spray Sunscreen SPF 30 is a sunscreen with Aloe Vera, Sunflower Oil, Baby Skin, Kids Skin, Sensitive Skin, Vegan Product, Mineral Active Ingredient, Water-Resistant Product"
output: hyponym("Aloe Vera"; "Ingredient"), hyponym("Sunflower Oil"; "Ingredient"), hyponym("Baby Skin"; "Effect"), hyponym("Kids Skin"; "Effect"), hyponym("Sensitive Skin"; "Effect"), hyponym("Vegan Product"; "Sustainable Product"),hyponym("Sustainable Product"; "Product"), hyponym("Mineral Active"; "Feature"), hyponym("Water-Resistant Product"; "Feature")

#7 
input: Hypernym: Glycerin Product, Hyponym: Cathy Doll L-Glutathione Magic Cream SPF 50 Whitening Sunscreen 138ml with Glycerin and whitening effect
output hyponym("Glycerin Product": "Feature"), hyponym("Cathy Doll L-Glutathione Magic Cream SPF 50 Whitening Sunscreen": "Product"), hyponym("Glycerin": "Ingredient"), hyponym("Whitening effect": "Feature")

#8
input: "The product is a sunscreen with Aloe Vera, Sunflower Oil, Baby Skin, Kids Skin, Sensitive Skin, Vegan Product, Mineral Active Ingredient, Water-Resistant Product"
output: hyponym("Aloe Vera"; "Ingredient"), hyponym("Sunflower Oil"; "Ingredient"), hyponym("Baby Skin"; "Effect"), hyponym("Kids Skin"; "Effect"), hyponym("Sensitive Skin"; "Effect"), hyponym("Vegan Product"; "Product"), hyponym("Mineral Active"; "Feature"), hyponym("Water-Resistant Product"; "Feature")

"""

prompt = ChatPromptTemplate(messages=[
    ("system", system_prompt),
    ("user", input_message)
], 
    input_variables=["Text"]
)



In [18]:
import pandas as pd
model = OllamaLLM(model="llama3.2",temperature=0)
hearst_df = pd.DataFrame(columns=["parent_asin","hypernym", "hyponym"])

def chain(prompt, model, input): 
    chain = prompt | model
    result = chain.invoke({f"Text": input})
    return result

# initialize dataframe
def store_results(hypernym, hyponym, df):
    df.append({"hypernym": hypernym, "hyponym": hyponym})
    return df

def extract_hearst_patterns(result,df, parent_asin):
    for line in result.split("\n"):
        if "hyponym" in line:
            # extract content inside the parentheses
            match = re.search(r'hyponym\("([^"]+)"\s*;\s*"([^"]+)"\)', line) 
            if match:
                hypernym, hyponym = match.groups()
                # Use pandas concat instead of append which is deprecated
                df = pd.concat([df, pd.DataFrame([{"parent_asin":parent_asin,"hypernym": hypernym, "hyponym": hyponym}])], ignore_index=True)
    return df


        
# try with iloc 1:10
for i in range(len(metadata_filtered_df.iloc[0:10])):
    print(f"Processing row {i+1} of {len(metadata_filtered_df)}")
    parent_asin = metadata_filtered_df.iloc[i].parent_asin
    result = chain(prompt, model, metadata_filtered_df.iloc[i].source_text)
    print(result)
    print("--------------------------------")
    # Pass hearst_df as an argument to extract_hearst_patterns to avoid UnboundLocalError
    hearst_df = extract_hearst_patterns(result,hearst_df, parent_asin)


Processing row 1 of 324
hyponym("Oil"; "Organic Sweet Almond Oil and Fractionated Coconut Oil Bundle for Hair and Skin"), 
hyponym("Bundle"; "Organic Sweet Almond Oil and Fractionated Coconut Oil Bundle for Hair and Skin"), 
hyponym("Brand"; "Shiny Leaf"), 
hyponym("Scent"; "Almond Oil and Fractionated Coconut"), 
hyponym("Item Form"; "Oil"), 
hyponym("Unit Count"; "32.00 Fl Oz"), 
hyponym("Number of Items"; "2"), 
hyponym("Package Dimensions"; "8.62 x 5.04 x 2.48 inches; 2.23 Pounds"), 
hyponym("UPC"; "781584950669"), 
hyponym("Store"; "Shiny Leaf"), 
hyponym("Categories"; "*"), 
hyponym("Price"; "None")
--------------------------------
Processing row 2 of 324
hyponym("Glass Bottle"; "Empty Brown Glass Spray Bottles2-Pack"), hyponym("Brand"; "Cherry"), hyponym("Material"; "Glass"), hyponym("Capacity"; "16 Ounces"), hyponym("Number of Items"; "2"), hyponym("Product Care Instructions"; "Hand Wash Only"), hyponym("Package Dimensions"; "9.09 x 6.46 x 3.35 inches"), hyponym("UPC"; "7915232